# Module 7 capstone, process twin: a calibrated furnace twin that stays in sync

**Digital Twins for Semiconductor Manufacturing · Module 7 · about 2 hours** · notebook version 2026-10-25

This is the **process twin** track of the capstone. You join two prepared components into one twin of the course's furnace and test it the way a twin should be tested before anyone relies on it:

- the **Deal-Grove model** of Module 3, calibrated on qualification runs, predicts the oxide thickness for an oxidation time;
- the **offset estimate** of Module 5, a Kalman filter, keeps that prediction in sync as the thermocouple drifts.

Before each production run the twin predicts the monitor-wafer thickness and recommends the oxidation time (recipe DRYOX-90: dry oxygen, 1000 °C, 80 to 100 minutes). You write two short functions that connect the components, one function that turns a test result into a verdict, and the **twin data sheet**: what the twin is for, what it uses, how it performed, where it must not be used, one risk, and what to do when confidence is low.

The test plan is fixed before any test runs:

| test | data | pass when |
|---|---|---|
| held-out | production runs 1 to 60 of the drift scenario, each predicted before it is measured | mean absolute error (MAE) at most 0.5 nm |
| baseline | the calibrated Deal-Grove model alone, on the same runs | (for comparison) |
| challenging | runs 31 to 60 of the failing-thermocouple scenario, a fixed window across the failure, which starts between runs 25 and 35 | MAE at most 0.5 nm |

Steps 1 to 8 use the **course example furnace (KEY 2026)**, so you can compare your numbers with the Module 7 page. Step 9 repeats the tests on your own furnace, with the KEY from Question 1 of the **Module 7 project quiz for the process twin**. All data are **simulated**. Run each code cell with **Shift + Enter**, in order.

## Step 0: start Python

The first time, Python can take up to a minute to start. The cell prints the versions of the libraries and of the course files next to this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import furnace, oxide_tools, r2r_tools, capstone, process_twin
from oxide_tools import predict
from capstone import mae_ci, blank_datasheet, show_options, show_datasheet
from process_twin import (qualification_runs, calibrate, kalman_update, next_time, run_twin, runs_between, plot_log,
                          TOLERANCE_NM, HELDOUT_RUNS, CHALLENGING_RUNS)

EXAMPLE_KEY = 2026
TOLERANCE = TOLERANCE_NM
print("Notebook version 2026-10-25")
print("NumPy", np.__version__, "| pandas", pd.__version__, "| furnace.py", furnace.__version__, "| oxide_tools.py", oxide_tools.__version__,
      "| r2r_tools.py", r2r_tools.__version__, "| capstone.py", capstone.__version__, "| process_twin.py", process_twin.__version__)

## Step 1: the prepared components

A twin in a fab is rarely written from scratch: it is assembled from models and tools that were each built and tested on their own. This track supplies these components:

| component | from | what it does |
|---|---|---|
| `qualification_runs(key)`, `calibrate(key)` | Module 3 | 12 qualification runs and the weighted Deal-Grove fit (SciPy's `curve_fit`) |
| `predict(fit, t)` | Module 3 | the calibrated thickness after `t` minutes, `predict(fit, t)["x"]`, with its bands |
| `kalman_update(d, P, y, Q, R)` | Module 5 | one update of the offset estimate; an observation of NaN keeps the estimate |
| `next_time(d_hat, target, B, BA, xi, limits)` | Module 5 | the oxidation time that aims the twin at the target, within 80 to 100 minutes |
| `run_twin(key, scenario, predict_next, offset_observation)` | new | the 60 production runs with the twin in the loop, using your two connections |
| `mae_ci(errors)` | Module 6 | the mean absolute error and the half-width of its 95 % interval |

What is missing is the **connection**: how the twin turns the calibrated model and the offset estimate into a prediction, and how it turns a measurement into an observation of the offset. That is your part, in Step 3.

## Step 2: calibrate the model on the qualification runs

Before production, the furnace runs 12 **qualification runs** of 60 to 126 minutes on bare wafers, with a freshly calibrated thermocouple, and one monitor wafer of each is measured (standard deviation 0.32 nm). `calibrate()` fits B and B/A to them with the Module 3 helper.

The fitted constants are not the textbook values: on bare wafers the oxide grows faster at first than Deal-Grove says (the thin-oxide regime of Module 3), and the two constants absorb that. The cell also prints χ²/dof, as in Module 3: near 1 the residuals are as large as the measurement noise; well above 1 the model leaves something out, here the thin-oxide regime. The calibrated model is good **inside** the qualified range of 60 to 126 minutes, which covers the recipe's 80 to 100 minutes. Outside that range nothing has tested it.

In [ ]:
qual = qualification_runs(EXAMPLE_KEY)
fit = calibrate(EXAMPLE_KEY)
setup = furnace.r2r_setup(EXAMPLE_KEY)
print(qual.to_string(index=False))
print()
print(f"B = {fit['B']:.2f} nm^2/min (+/- {fit['se_B']:.2f}),  B/A = {fit['BA']:.4f} nm/min (+/- {fit['se_BA']:.4f})")
print(f"chi2 / dof = {fit['chi2']:.1f} / {fit['dof']} = {fit['chi2'] / fit['dof']:.2f}:  the residuals are about "
      f"{np.sqrt(fit['chi2'] / fit['dof']):.1f} times the stated measurement noise")
print(f"target thickness of recipe DRYOX-90: {setup['target_nm']} nm")

## Step 3: your task, connect the components

The twin's prediction for a run of `t_min` minutes is the calibrated Deal-Grove thickness plus the current offset estimate `d_hat` (Module 5: the offset is how far the furnace now sits from the model):

prediction = `predict(fit, t_min)["x"][0]` + `d_hat`

After the run, the monitor wafer gives one **observation of the offset**: the measured thickness minus the calibrated model's thickness for the time that ran (without `d_hat`: the observation is of the offset itself). When a run has no measurement, `measured_nm` is NaN, and so is the observation; the Kalman update then keeps its estimate.

**Your task:** write `predict_next(t_min, fit, d_hat)` and `offset_observation(measured_nm, t_min, fit)`. Each returns one number in nm.

In [ ]:
def predict_next(t_min, fit, d_hat):
    """The twin's prediction (nm) of the monitor-wafer thickness for a run of t_min minutes."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete predict_next() first.")


def offset_observation(measured_nm, t_min, fit):
    """The offset observed in one run: measured minus the calibrated model (nm); NaN when there is no measurement."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete offset_observation() first.")

In [ ]:
print(predict_next(90.0, fit, 0.0), "  (the calibrated model at 90 minutes)")
print(predict_next(90.0, fit, -1.0), "  (should be 1 nm less than the line above)")
print(offset_observation(60.0, 90.0, fit), "  (60 nm measured, minus the model at 90 minutes)")
print(offset_observation(np.nan, 90.0, fit), "  (should be nan: no measurement)")

## Step 4: verify the connected twin

**Verification and validation are different questions.** Verification asks whether the code computes what the model says. Validation asks whether the model's answers are close enough to the real system for the use you intend. (These are the definitions the National Academies use for digital twins, from the National Research Council's 2012 report.) Verification comes first: a validation result means little if the code does not do what you think it does. The tests below compare the connected twin with answers you can work out exactly.

Four tests: (1) when the twin aims at the target and its time is inside the limits, its prediction is the target exactly; (2) if a run measures exactly what the twin predicted, the observed offset equals the estimate; (3) a run with no measurement leaves the estimate unchanged; (4) the predictions lie in a plausible range for the recipe.

In [ ]:
target, lim, xi = setup["target_nm"], setup["t_limits"], furnace.R2R_XI_NM
ok1 = True
for d in (-2.0, -0.5, 0.0, 1.5):
    t = next_time(d, target, fit["B"], fit["BA"], xi, lim)
    if lim[0] < t < lim[1]:
        ok1 &= abs(predict_next(t, fit, d) - target) < 1e-6
ok2 = all(abs(offset_observation(predict_next(t, fit, d), t, fit) - d) < 1e-9 for t in (82.0, 90.0, 98.0) for d in (-1.2, 0.0, 0.8))
d_new, P_new = kalman_update(0.7, 0.05, offset_observation(np.nan, 90.0, fit), 0.01, 0.16)
ok3 = d_new == 0.7 and abs(P_new - 0.06) < 1e-12
ok4 = all(50.0 < predict_next(t, fit, 0.0) < 75.0 for t in (80.0, 90.0, 100.0))
for i, ok in enumerate((ok1, ok2, ok3, ok4), 1):
    print(f"verification test {i}:", "ok" if ok else "FAILED: check Step 3")

## Step 5: validate on held-out runs

Now the twin runs production: 60 runs of the **drift** scenario of Module 5, which it has never seen. Before each run it recommends the oxidation time and predicts the monitor wafer; after the run the measurement updates the offset estimate. Each run is predicted **before** its measurement is used, so every error is an honest out-of-sample error, even though the twin keeps learning. One run has no measurement and adds no error.

The MAE is a mean over 59 runs, and those runs are a sample: `mae_ci()` gives the half-width of its 95 % interval, t(0.975, n − 1) × s / √n, as in Module 6. The interval is approximate: it treats the runs as independent, but the offset estimate carries part of each run's error into the next runs, so neighbouring errors can be correlated, most of all while the twin adapts after a change.

In the plot, the shaded band shows the size of the tolerance, for scale. The verdict compares the MAE of the whole test window, printed above the lower panel with its interval, with the tolerance: a single run outside the band does not fail the twin, and an MAE within the tolerance does not make every run acceptable.

In [ ]:
log_drift = run_twin(EXAMPLE_KEY, "drift", predict_next, offset_observation)
plot_log(log_drift, title=f"Process twin, example furnace (KEY {EXAMPLE_KEY}), drift scenario", window=HELDOUT_RUNS, window_name="held-out")
held = runs_between(log_drift, HELDOUT_RUNS)
h_mae, h_hw, h_n = mae_ci(held["error_nm"])
print(f"held-out: MAE {h_mae:.3f} nm, 95 % interval +/- {h_hw:.3f} nm, over {h_n} measured runs (tolerance {TOLERANCE} nm)")

## Step 6: your task, `verdict()`

A test gives an error with an interval, not a single number, and the **tolerance** was set before the test, from the twin's intended use: here the mean absolute error must be at most **0.5 nm**. Compare the whole interval with the tolerance:

- **meets**: the whole interval lies at or below the tolerance, MAE + half-width ≤ tolerance;
- **misses**: the whole interval lies above the tolerance, MAE − half-width > tolerance;
- **insufficient evidence**: the interval contains the tolerance, so this test cannot tell. More test data would narrow the interval.

"Insufficient evidence" is an honest result, not a failure of the method. It is better than a "meets" that the data do not support.

The 95 % interval from `mae_ci()` is **approximate**. It treats the test units (runs or weeks) as independent, and neighbouring units can be correlated: when an error carries over from one unit to the next, the true uncertainty is larger than the computed interval, most of all after a change, when the errors move together for several units. Read a verdict close to the tolerance with that in mind.

**Your task:** write `verdict(mae, half_width, tolerance)`. Return one of the three strings exactly as written here: `"meets"`, `"misses"` or `"insufficient evidence"`.

In [ ]:
def verdict(mae, half_width, tolerance):
    """"meets", "misses" or "insufficient evidence": the 95 % interval of the MAE, mae +/- half_width, against the tolerance."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete verdict() first.")

In [ ]:
print(verdict(0.30, 0.10, 0.5), "  (should be meets: the interval runs from 0.20 to 0.40)")
print(verdict(0.45, 0.10, 0.5), "  (should be insufficient evidence: 0.35 to 0.55 contains 0.5)")
print(verdict(0.90, 0.20, 0.5), "  (should be misses: 0.70 to 1.10 lies above 0.5)")
print()
print("held-out test of the example:", verdict(h_mae, h_hw, TOLERANCE))

## Step 7: compare with the baseline

A twin should beat the simplest thing that could work. The **baseline** here is the calibrated Deal-Grove model alone, with no offset estimate: Module 3 without Module 5. `run_twin()` records its prediction for the same runs and the same oxidation times.

If the twin did not beat its baseline, that would be a valid finding too: the simpler model would be enough.

In [ ]:
b_mae, b_hw, b_n = mae_ci(held["baseline_error_nm"])
print(f"baseline (Deal-Grove alone): MAE {b_mae:.3f} nm, +/- {b_hw:.3f} nm")
print(f"twin:                        MAE {h_mae:.3f} nm, +/- {h_hw:.3f} nm")
print(f"the twin's error is {h_mae / b_mae:.0%} of the baseline's")

## Step 8: the challenging condition

Every validation needs one condition that is harder than normal production. Here it is the **failing thermocouple** of Module 5: from a run between 25 and 35 its bias grows several times faster. The offset estimate was tuned for slow drift (Q = 0.01 nm² per run), so it now lags behind. The test uses a fixed window, runs 31 to 60 of the failing scenario. The failure starts between run 25 and run 35, depending on the furnace, so for some furnaces the first runs of the window come before the failure: the window spans the transition and the operation after it. The challenge is built in on purpose, and the notebook says so: a twin is only trusted where it has been tested.

In [ ]:
log_fail = run_twin(EXAMPLE_KEY, "failing", predict_next, offset_observation)
plot_log(log_fail, title=f"Process twin, example furnace (KEY {EXAMPLE_KEY}), failing thermocouple", window=CHALLENGING_RUNS, window_name="challenging")
chal = runs_between(log_fail, CHALLENGING_RUNS)
c_mae, c_hw, c_n = mae_ci(chal["error_nm"])
print(f"challenging: MAE {c_mae:.3f} nm, +/- {c_hw:.3f} nm, over {c_n} runs:", verdict(c_mae, c_hw, TOLERANCE))

## Step 9: your own furnace

Open the **Module 7 project quiz for the process twin** in Canvas. Question 1 gives you a **KEY**, a whole number from 100 to 999. Type it below in place of `None` and run the cell. It repeats Steps 5 to 8 on **your furnace** with **your functions** and prints the numbers for the data sheet.

In [ ]:
KEY = None      # <- your KEY from Question 1 of the project quiz, for example KEY = 417

if KEY is None:
    print("Type your KEY in place of None, then run this cell again.")
else:
    my_drift = run_twin(KEY, "drift", predict_next, offset_observation)
    my_fail = run_twin(KEY, "failing", predict_next, offset_observation)
    my_held = runs_between(my_drift, HELDOUT_RUNS)
    results = {"heldout": mae_ci(my_held["error_nm"]),
               "challenging": mae_ci(runs_between(my_fail, CHALLENGING_RUNS)["error_nm"]),
               "baseline": mae_ci(my_held["baseline_error_nm"])}
    print(f"Your furnace, KEY {KEY} (tolerance {TOLERANCE} nm):")
    for name, (m, hw, n) in results.items():
        v = verdict(m, hw, TOLERANCE) if name != "baseline" else "(for comparison)"
        print(f"  {name:12s} MAE {m:.3f} nm   +/- {hw:.3f}   n = {n:2d}   {v}")
    plot_log(my_drift, title=f"Your furnace (KEY {KEY}), drift scenario", window=HELDOUT_RUNS, window_name="held-out")

## Step 10: the twin data sheet

A twin that others will use needs a label, as a dataset or a model does (datasheets for datasets, Gebru et al. 2021; model cards, Mitchell et al. 2019). The **twin data sheet** has six fields. Fields 1, 2, 5 and 6 are fixed choices, field 3 holds your numbers from Step 9, and field 4 is a list of choices. The cell below prints the choices for this track.

How to fill in field 4, the conditions where the predictions should **not** be used:

- list every condition **outside** the range the twin was calibrated and tested in: nothing supports a claim there;
- list each tested condition whose verdict was **not** "meets" (that is, "misses" or "insufficient evidence");
- do **not** list a condition that your tests included and that met the tolerance: your results support it.

Field 5 needs a risk that this twin's data path really has, and a mitigation that reduces that risk (NIST IR 8356 discusses the security and trust risks of digital twins). Field 6 is what the user should do when the twin's confidence is low.

The check cell in Step 11 reads this data sheet and compares it with your results: the numbers must match Step 9 (3 decimals), the verdicts must follow your `verdict()` rule, and the choices must be consistent with what you found.

In [ ]:
show_options("process")

In [ ]:
datasheet = blank_datasheet()
datasheet["intended_use"] = None           # 1. one letter, for example "A"
datasheet["data_sources"] = None           # 2. one letter
datasheet["heldout_mae"] = None            # 3. your numbers from Step 9, 3 decimals
datasheet["heldout_verdict"] = None        #    "meets", "misses" or "insufficient evidence"
datasheet["challenging_mae"] = None
datasheet["challenging_verdict"] = None
datasheet["baseline_mae"] = None
datasheet["do_not_use"] = None             # 4. a list of letters, for example ["A", "D"]
datasheet["risk_mitigation"] = None        # 5. one letter
datasheet["low_confidence_action"] = None  # 6. one letter
datasheet["note"] = ""                     # optional, not checked

show_datasheet(datasheet, "process", results=globals().get("results"), tolerance=TOLERANCE, key=KEY, version="2026-10-25")

## Step 11: check your work

The cell below tests your functions on cases you cannot see, recomputes the validation results for your KEY with the course's reference functions, and compares your data sheet with them. If everything passes, it prints your **verification value**, a code made from your KEY and the three numbers of your data sheet. Type it into Question 1 of the **Module 7 project quiz for the process twin**.

If it says **NOT PASSED**, read the message: it tells you what to fix. Fix it, run the cells you changed, then run this cell again.

In [ ]:
# The course's check code is in dt_m7_check.py, next to this notebook.
# It is kept out of the notebook so that the course's reference solutions are not on screen.
from dt_m7_check import check_process_twin

passed, messages, value = check_process_twin(predict_next, offset_observation, verdict, datasheet, KEY)
for line_text in messages:
    print(line_text)
if passed:
    print()
    print(f"Verification value for KEY = {KEY}: {value:.3f}")
    print("Type this number into Question 1 of the Module 7 project quiz for the process twin.")

## What the data sheet says about the twin

You built a twin from parts that each worked on their own, verified that the connected code does what the model says, and validated it on data it had not seen, against a tolerance set before the test, next to a baseline and under one challenging condition. The data sheet records the result for whoever uses the twin next: what it is for, what it uses, how well it did, where it must not be used, one risk, and what to do when its confidence is low.

**What the process twin showed.** Kept in sync, the calibrated model predicts the monitor wafer to within its tolerance during normal drift, several times better than the model alone. When the thermocouple fails, the same twin lags behind, because its filter was tuned for slow drift: that condition goes on the data sheet as one where its predictions must not be used, and Module 5's alarm and maintenance rule take over.

**Validation is not done once.** The physical system keeps changing: thermocouples age, recipes change, tools are replaced. The National Academies' 2024 report on digital twins concludes that their verification, validation and uncertainty quantification must be a continual process, revisited as the physical system, the data and the use change. A data sheet carries a date for that reason.

**Claim only what the data support.** On simulated data, a result shows how well the twin recovers the course's simulator. It does not validate a twin of a real furnace or a real line.

To keep a copy of this notebook, choose **File > Download**.